In [2]:
# ================================================================
# LLAMAINDEX RAG DEMONSTRATION USING GEMINI
# SINGLE GOOGLE COLAB / JUPYTER NOTEBOOK CELL
# ================================================================

# ---------------------------------------------------------------
# 1. Install required packages
# ---------------------------------------------------------------

import sys
import subprocess
import os
from getpass import getpass

required_packages = [
    "llama-index",
    "llama-index-llms-gemini",
    "llama-index-embeddings-gemini",
    "google-genai"
]

print("Installing required packages...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        *required_packages
    ]
)

print("Packages installed successfully.\n")


Installing required packages...
Packages installed successfully.



In [4]:
# ---------------------------------------------------------------
# 2. Import LlamaIndex and Gemini components
# ---------------------------------------------------------------

from llama_index.core import (
    Document,
    Settings,
    VectorStoreIndex
)

from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding

In [5]:
# ---------------------------------------------------------------
# 3. Read the Gemini API key securely
# ---------------------------------------------------------------

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
#client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key

In [6]:
# ---------------------------------------------------------------
# 4. Configure Gemini LLM
# ---------------------------------------------------------------

gemini_llm = Gemini(
    model="models/gemini-2.5-flash",
    api_key=api_key,
    temperature=0
)


# ---------------------------------------------------------------
# 5. Configure Gemini embedding model
# ---------------------------------------------------------------

gemini_embedding_model = GeminiEmbedding(
    model_name="models/gemini-embedding-001",
    api_key=api_key
)


# Apply the models globally in LlamaIndex.
Settings.llm = gemini_llm
Settings.embed_model = gemini_embedding_model

# Configure document chunking.
Settings.chunk_size = 256
Settings.chunk_overlap = 30


# ---------------------------------------------------------------
# 6. Create sample knowledge-base documents
# ---------------------------------------------------------------

knowledge_base = [

    """
    ABC Internet Services: Router Troubleshooting

    If the router's red light is blinking, restart the router
    and verify that the fibre-optic cable is securely connected.

    Switch off the router, wait for approximately thirty seconds,
    and then switch it on again.

    If the red light continues blinking after the restart,
    contact the technical-support department.
    """,

    """
    ABC Internet Services: Slow Internet Connection

    If the internet connection is slow, first check the Wi-Fi
    signal strength.

    Move closer to the router and disconnect unnecessary devices.

    Restart the modem and router.

    Customers should also check whether large downloads or
    software updates are consuming the available bandwidth.
    """,

    """
    ABC Internet Services: Password Management

    Customers can reset their account password by selecting
    Forgot Password on the login page.

    A password-reset link will be sent to the customer's
    registered email address.

    The link expires after thirty minutes.

    The new password should contain at least eight characters,
    one uppercase letter, one lowercase letter and one number.
    """,

    """
    ABC Internet Services: Warranty Policy

    Warranty replacement is available within one year from
    the original purchase date.

    The customer must provide the purchase invoice and the
    serial number of the device.

    Physical damage, water damage and damage caused by
    unauthorized repairs are not covered by the warranty.
    """,

    """
    ABC Internet Services: Billing Support

    For billing problems, customers should contact the accounts
    department and provide their invoice number.

    Billing complaints may include incorrect charges, duplicate
    payments, failed transactions and missing payment receipts.

    The accounts department normally responds within two
    working days.
    """,

    """
    ABC Internet Services: Support Hours

    Technical support is available from Monday to Saturday,
    between 9:00 a.m. and 6:00 p.m.

    Billing support is available from Monday to Friday,
    between 10:00 a.m. and 5:00 p.m.

    Emergency network-outage complaints can be registered
    through the customer-support portal at any time.
    """
]


# Convert the text strings into LlamaIndex Document objects.
documents = []

for document_number, text in enumerate(
    knowledge_base,
    start=1
):
    document = Document(
        text=text.strip(),
        metadata={
            "document_number": document_number,
            "source": "ABC Internet Services Knowledge Base"
        }
    )

    documents.append(document)

print(
    f"Created {len(documents)} "
    "knowledge-base documents."
)


/tmp/ipykernel_3423/1662713058.py:5: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  gemini_llm = Gemini(
/tmp/ipykernel_3423/1662713058.py:16: DeprecationWarning: Call to deprecated class GeminiEmbedding. (Should use `llama-index-embeddings-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/embeddings/google_genai/Support for this package will be discontinued as of v0.4.2) -- Deprecated since version 0.4.2.
  gemini_embedding_model = GeminiEmbedding(


Created 6 knowledge-base documents.


In [7]:
# ---------------------------------------------------------------
# 7. Build the vector index
# ---------------------------------------------------------------

print(
    "Creating Gemini embeddings and building "
    "the vector index..."
)

index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True
)

print("Vector index created successfully.\n")


# ---------------------------------------------------------------
# 8. Create the RAG query engine
# ---------------------------------------------------------------

query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact",
    llm=gemini_llm
)


# ---------------------------------------------------------------
# 9. Run one demonstration question
# ---------------------------------------------------------------

demo_question = (
    "What should I do if the router red light is blinking?"
)

print("=" * 75)
print("DEMONSTRATION QUESTION")
print("=" * 75)
print(demo_question)

demo_response = query_engine.query(
    demo_question
)

print("\nANSWER")
print("-" * 75)
print(demo_response)


# ---------------------------------------------------------------
# 10. Display retrieved source nodes
# ---------------------------------------------------------------

print("\nRETRIEVED SOURCES")
print("-" * 75)

for source_number, source_node in enumerate(
    demo_response.source_nodes,
    start=1
):
    similarity_score = source_node.score

    if similarity_score is None:
        score_text = "Not available"
    else:
        score_text = f"{similarity_score:.4f}"

    metadata = source_node.node.metadata
    retrieved_text = (
        source_node.node
        .get_content()
        .strip()
    )

    print(f"\nSource {source_number}")
    print(f"Similarity score: {score_text}")

    print(
        "Document number:",
        metadata.get(
            "document_number",
            "Unknown"
        )
    )

    print("Retrieved text:")
    print(retrieved_text[:600])



Creating Gemini embeddings and building the vector index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/6 [00:00<?, ?it/s]

Vector index created successfully.

DEMONSTRATION QUESTION
What should I do if the router red light is blinking?

ANSWER
---------------------------------------------------------------------------
If the router's red light is blinking, you should first restart the router and ensure the fibre-optic cable is securely connected. To restart, switch off the router, wait for about thirty seconds, and then switch it back on. If the red light continues to blink after this restart, contact the technical-support department.

RETRIEVED SOURCES
---------------------------------------------------------------------------

Source 1
Similarity score: 0.8437
Document number: 1
Retrieved text:
ABC Internet Services: Router Troubleshooting

    If the router's red light is blinking, restart the router
    and verify that the fibre-optic cable is securely connected.

    Switch off the router, wait for approximately thirty seconds,
    and then switch it on again.

    If the red light continues blinking 

In [8]:

# ---------------------------------------------------------------
# 11. Start the interactive RAG application
# ---------------------------------------------------------------

print("\n" + "=" * 75)
print("GEMINI + LLAMAINDEX CUSTOMER-SUPPORT RAG")
print("=" * 75)

print(
    """
Ask questions about:

- Router problems
- Slow internet
- Password reset
- Warranty
- Billing
- Support hours

Type 'exit' to stop.
"""
)


while True:

    question = input("Your question: ").strip()

    if question.lower() in {
        "exit",
        "quit",
        "stop"
    }:
        print(
            "\nLlamaIndex RAG application stopped."
        )
        break

    if not question:
        print(
            "Please enter a valid question.\n"
        )
        continue

    try:
        response = query_engine.query(
            question
        )

        print("\nANSWER")
        print("-" * 75)
        print(response)

        print("\nRETRIEVED SOURCES")
        print("-" * 75)

        if not response.source_nodes:
            print(
                "No source nodes were returned."
            )

        for source_number, source_node in enumerate(
            response.source_nodes,
            start=1
        ):
            similarity_score = source_node.score

            if similarity_score is None:
                score_text = "Not available"
            else:
                score_text = (
                    f"{similarity_score:.4f}"
                )

            metadata = source_node.node.metadata

            source_text = (
                source_node.node
                .get_content()
                .strip()
                .replace("\n", " ")
            )

            print(
                f"\nSource {source_number} "
                f"— similarity score: {score_text}"
            )

            print(
                "Document number:",
                metadata.get(
                    "document_number",
                    "Unknown"
                )
            )

            print(source_text[:400])

        print("\n" + "=" * 75 + "\n")

    except KeyboardInterrupt:
        print(
            "\nApplication interrupted by the user."
        )
        break

    except Exception as error:
        print(
            "\nThe query could not be completed."
        )

        print(
            f"{type(error).__name__}: {error}\n"
        )


GEMINI + LLAMAINDEX CUSTOMER-SUPPORT RAG

Ask questions about:

- Router problems
- Slow internet
- Password reset
- Warranty
- Billing
- Support hours

Type 'exit' to stop.

Your question: My internet is slow, why?

ANSWER
---------------------------------------------------------------------------
A slow internet connection can be due to several factors. You should first check your Wi-Fi signal strength and consider moving closer to your router. Disconnecting any unnecessary devices from your network can also help. Additionally, restarting both your modem and router is a common troubleshooting step. Finally, ensure that large downloads or ongoing software updates are not consuming your available bandwidth.

RETRIEVED SOURCES
---------------------------------------------------------------------------

Source 1 — similarity score: 0.8272
Document number: 2
ABC Internet Services: Slow Internet Connection      If the internet connection is slow, first check the Wi-Fi     signal strength.